In [ ]:
"""This notebook matches 1.5 micron scans to 3 micron scans for 3 samples: OFF-211, K270, K229
(FWZ-1-1 1.5 scan was poor quality)
once I match the grains from the 1.5 and 3 micron scans, I will calculate roundness for both
and plot measurement error distributions
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import glob
mpl.rcParams['figure.dpi'] = 300

# sample info

sample_id = 'K229'

# path to directory of labeled volumes
path = '../../../processed_data/labeled/'

# load both volumes -- hr = hi res; lr = low res
hr_file = glob.glob(path + sample_id + '_1p5um*')
lr_file = glob.glob(path + sample_id + '_3um*')

if len(hr_file) != 1 or len(lr_file) != 1:
    raise FileNotFoundError("Expected exactly one matching file for each scan.")

hr_vol = np.load(hr_file[0]) # 1.5 micron data
lr_vol = np.load(lr_file[0]) # 3 micron data

print(hr_vol.shape)
print(lr_vol.shape)


In [ ]:
# also load in raw data for the 1.5 to get the size of the cylindrical field
# of view

raw_hr = np.load('../../../processed_data/raw/'+sample_id+'_1p5um.npy')

flat_raw_hr = raw_hr.max(axis=0)

cylinder = (flat_raw_hr != flat_raw_hr[0,0]).astype(np.uint8)

# calculate diameter of circle in both x and y axes
Dx = cylinder.sum(axis=0).max()
Dy = cylinder.sum(axis=1).max()

Dx = np.int64(Dx)
Dy = np.int64(Dy)

if np.abs(Dx - Dy) > 1:
    print('not perfect circle! Inspection needed')
else:
    diam = Dx

print(diam)
# plt.imshow(cylinder, 'gray')
plt.imshow(flat_raw_hr, 'gray')

In [ ]:
# Show max-intensity projections along axis 0 for hr_vol and lr_vol side by side
hr_max = (hr_vol > 0).max(axis=0)
lr_max = (lr_vol > 0).max(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(hr_max, cmap='nipy_spectral')
axes[0].set_title(f'hr_vol max(axis=0)\nshape={hr_max.shape}')
axes[0].axis('off')

axes[1].imshow(lr_max, cmap='nipy_spectral')
axes[1].set_title(f'lr_vol max(axis=0)\nshape={lr_max.shape}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# downsample hi res vol to make it possible to match
ds_hr_vol = hr_vol[::2,::2,::2]
ds_hr_max = ds_hr_vol.max(axis=0)

# make a mask that is an empty cylinder
h, w = lr_vol.shape[1], lr_vol.shape[2]
cy, cx = 860, 240 # EDIT THIS FOR EACH SAMPLE
yy, xx = np.ogrid[:h, :w]

r = (diam / 4) - 5
mask_2d = (((yy - cy) ** 2 + (xx - cx) ** 2) > r**2).astype(np.uint8)

# 3D mask matching ds_hr_vol (same 2D mask for every z-slice)
mask = np.broadcast_to(mask_2d, lr_vol.shape).copy()

# now show mask on lr_vol

xor_overlay = np.logical_xor(lr_max.astype(bool), mask_2d.astype(bool))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(ds_hr_max > 0, cmap='gray')
axes[0].set_title('ds_hr_max')

axes[1].imshow(xor_overlay[350:,:700], cmap='magma') # EDIT SLICES FOR EACH SAMPLE
axes[1].set_title('XOR: lr_max ^ mask_2d')

plt.tight_layout()
plt.show()


In [ ]:
from skimage.measure import label, regionprops

# label binary volume
labeled = label(np.logical_or((lr_vol > 0),mask))
regions = regionprops(labeled)

mask2 = (labeled != regions[0].label)

# apply mask
lr_center_vol = mask2 * lr_vol
crop_lr_center_max = (lr_center_vol[:,400:,:700]).max(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(ds_hr_max > 0, cmap='gray')
axes[0].set_title('ds_hr_max')

axes[1].imshow(crop_lr_center_max > 0, cmap='gray')
axes[1].set_title('lr_center_max')

In [ ]:
import numpy as np
from scipy.ndimage import center_of_mass

# x / y shifts to hr_com
shifts = np.array([0, 30]) # CHANGE FOR EACH SAMPLE

# Nonzero labels
hr_labels = np.unique(ds_hr_max)
hr_labels = hr_labels[hr_labels != 0]

lr_labels = np.unique(crop_lr_center_max)
lr_labels = lr_labels[lr_labels != 0]

# Centers of mass
hr_com = np.array(
    center_of_mass(ds_hr_max > 0, ds_hr_max, hr_labels)
) + shifts

lr_com = np.array(
    center_of_mass(crop_lr_center_max > 0, crop_lr_center_max, lr_labels)
)

# Find nearest HR label for each LR label
dists = np.sum((lr_com[:, None] - hr_com[None]) ** 2, axis=-1)
nearest_hr = hr_labels[np.argmin(dists, axis=1)]

# N x 2 array: [LR label, nearest HR label]
label_mapping = np.column_stack((lr_labels, nearest_hr))

# label_mapping[:, 0]  # LR labels
# label_mapping[:, 1]  # corresponding nearest HR labels

# plot center of masses for dr_hr_max in blue, for lr_center_max in red
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(hr_com[:, 1], hr_com[:, 0], c='blue', s=10, label='ds_hr_max COM')
ax.scatter(lr_com[:, 1], lr_com[:, 0], c='red', s=10, label='crop_lr_center_max COM')

ax.set_title('Center of Mass Comparison')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
dir = './3um_to_1.5um_label_maps/'
fn = dir + sample_id + 'label_map.npy'
np.save(fn, label_mapping)
# N x 2 array